In [64]:
import torch
import torch.nn as nn 
import torch.optim as optim
import numpy as np
from collections import Counter
from torch.utils.data import Dataset,DataLoader
import nltk
from nltk.tokenize import word_tokenize 

In [65]:
document = """ The Last Train Journey
On a cold winter evening, Daniel arrived at the old, abandoned train station on the outskirts of Blackwood. Snow was beginning to fall, dusting the rusted tracks with a silent, pristine white.
He clutched the tarnished brass pocket watch his grandfather had left him, its steady, rapid ticking the only sound in the freezing air. According to the final letter, the last train would arrive tonight at exactly midnight—a train that was no longer on any modern timetable.
As the wind howled through the skeletal rafters of the platform, a distant, mournful whistle echoed down the valley. A single amber headlight pierced the thick fog. With a screech of ancient iron, a steam locomotive pulled up to the platform, its windows glowing with a warm, amber light. 
The heavy door hissed open. Daniel took a deep breath, stepped aboard, and watched the station vanish into the snowy dark, embarking on a journey into the secrets of his family's past.
Inside the carriage, velvet seats lined the walls and a conductor in a faded uniform stood waiting without a ticket in his hand. He nodded once at Daniel and whispered, "Your grandfather rode this line every winter until the day he disappeared."
The train lurched forward, and the world outside the window began to change. Snow gave way to golden fields, then to a bustling town square that Daniel recognized from old photographs in the attic.
Daniel walked down the narrow aisle and found a compartment where a young man sat reading a letter by candlelight. The man's face was unmistakable—it was his grandfather, decades younger, with the same watch chain glinting at his waistcoat.
Before Daniel could speak, the train slowed again and stopped at a station that should not have existed. A wooden sign read Blackwood Crossing, and beneath it, painted in fading letters, were the words: All debts must be paid.
His grandfather looked up from the letter and met Daniel's eyes with a sorrowful smile. "You should not have come tonight," he said softly, "but since you are here, you must choose whether to leave the past buried or carry its truth into the morning."
Daniel sat across from him as the watch in his pocket slowed its ticking, matching the rhythm of the train's wheels on the iron rails. Outside, the snow began to fall once more, and the amber light in the windows grew brighter, as if the journey itself were remembering every mile it had ever traveled.
When the train finally reached the edge of the valley, Daniel understood at last why the timetable had vanished and why only one passenger was ever meant to board. He folded his grandfather's letter carefully, stepped back onto the platform, and walked home through the dawn with a story the world would never believe and a secret he would finally keep.
"""


In [66]:

nltk.download('punkt')

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\atiku\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\atiku\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [67]:

tokens = word_tokenize(document.lower())

print(tokens)

['the', 'last', 'train', 'journey', 'on', 'a', 'cold', 'winter', 'evening', ',', 'daniel', 'arrived', 'at', 'the', 'old', ',', 'abandoned', 'train', 'station', 'on', 'the', 'outskirts', 'of', 'blackwood', '.', 'snow', 'was', 'beginning', 'to', 'fall', ',', 'dusting', 'the', 'rusted', 'tracks', 'with', 'a', 'silent', ',', 'pristine', 'white', '.', 'he', 'clutched', 'the', 'tarnished', 'brass', 'pocket', 'watch', 'his', 'grandfather', 'had', 'left', 'him', ',', 'its', 'steady', ',', 'rapid', 'ticking', 'the', 'only', 'sound', 'in', 'the', 'freezing', 'air', '.', 'according', 'to', 'the', 'final', 'letter', ',', 'the', 'last', 'train', 'would', 'arrive', 'tonight', 'at', 'exactly', 'midnight—a', 'train', 'that', 'was', 'no', 'longer', 'on', 'any', 'modern', 'timetable', '.', 'as', 'the', 'wind', 'howled', 'through', 'the', 'skeletal', 'rafters', 'of', 'the', 'platform', ',', 'a', 'distant', ',', 'mournful', 'whistle', 'echoed', 'down', 'the', 'valley', '.', 'a', 'single', 'amber', 'headli

In [68]:
Counter(tokens).keys()

# build vocab

vocab = { '<unk>':0}

for token in Counter(tokens).keys():
    if token not in vocab:
        vocab[token] = len(vocab) 

vocab

{'<unk>': 0,
 'the': 1,
 'last': 2,
 'train': 3,
 'journey': 4,
 'on': 5,
 'a': 6,
 'cold': 7,
 'winter': 8,
 'evening': 9,
 ',': 10,
 'daniel': 11,
 'arrived': 12,
 'at': 13,
 'old': 14,
 'abandoned': 15,
 'station': 16,
 'outskirts': 17,
 'of': 18,
 'blackwood': 19,
 '.': 20,
 'snow': 21,
 'was': 22,
 'beginning': 23,
 'to': 24,
 'fall': 25,
 'dusting': 26,
 'rusted': 27,
 'tracks': 28,
 'with': 29,
 'silent': 30,
 'pristine': 31,
 'white': 32,
 'he': 33,
 'clutched': 34,
 'tarnished': 35,
 'brass': 36,
 'pocket': 37,
 'watch': 38,
 'his': 39,
 'grandfather': 40,
 'had': 41,
 'left': 42,
 'him': 43,
 'its': 44,
 'steady': 45,
 'rapid': 46,
 'ticking': 47,
 'only': 48,
 'sound': 49,
 'in': 50,
 'freezing': 51,
 'air': 52,
 'according': 53,
 'final': 54,
 'letter': 55,
 'would': 56,
 'arrive': 57,
 'tonight': 58,
 'exactly': 59,
 'midnight—a': 60,
 'that': 61,
 'no': 62,
 'longer': 63,
 'any': 64,
 'modern': 65,
 'timetable': 66,
 'as': 67,
 'wind': 68,
 'howled': 69,
 'through': 70,
 

In [69]:
input_sentences = document.split('\n')

In [70]:
def text_to_indices(sentence,vocab):
    numerical_sentence = []
    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])
        else:
            numerical_sentence.append(vocab['<unk>'])    

In [71]:
input_numerical_sentences = []
for sentence in input_sentences:
    input_numerical_sentences.append(text_to_indices(word_tokenize(sentence.lower()), vocab))
